# Phase 11 — Physics Feature Benchmark (Ablation)
**RealCentric Generator-Agnostic Deepfake Detection**

Confirmatory ablation study isolating precise AUC contributions of the physics-based handcrafted blocks sequentially after robust multi-domain training. 
Tests hypothesis: CNN features dominate generalisation under multi-domain training (~0.99 FF++ AUC), while physics sub-blocks (Extended HC) provide a modest but measurable independent robustness signal.

**Models Tested:**
1. **Full [0:974]**: The multi-trained pipeline baseline (NB09)
2. **Base [0:718]**: Handcrafted + CNN backbone
3. **CNN-Only [206:718]**: Only latent spatial/semantic signals
4. **Handcrafted-Only [0:206]**: The physics-only baseline (Mathematical Formulation claim)
5. **Extended [718:974]**: Only the 256 meaningful dims of the extended physics block (zeros trimmed)


## Step 1 — Load Features

In [1]:
import sys, numpy as np, time
import matplotlib.pyplot as plt
sys.path.insert(0, '/data/mpstme-naman/deepfake_detection')
from pathlib import Path

BASE     = Path('/data/mpstme-naman/deepfake_detection')
FEAT_DIR = BASE / 'data' / 'features'
CKPT_DIR = BASE / 'checkpoints'
RES_DIR  = BASE / 'results'

# Multi-dataset splits
Z_tr = np.load(FEAT_DIR/'Z_train_multi.npy')
y_tr = np.load(FEAT_DIR/'y_train_multi.npy')
Z_val = np.load(FEAT_DIR/'Z_val_multi.npy')
y_val = np.load(FEAT_DIR/'y_val_multi.npy')
Z_test = np.load(FEAT_DIR/'Z_test_multi.npy')
y_test = np.load(FEAT_DIR/'y_test_multi.npy')

# Cross-domain target datasets
Z_cd = np.load(FEAT_DIR/'Z_test.npy'); y_cd = np.load(FEAT_DIR/'y_test.npy')
Z_ff = np.load(FEAT_DIR/'Z_ff.npy'); y_ff = np.load(FEAT_DIR/'y_ff.npy')
Z_sd = np.load(FEAT_DIR/'Z_sd.npy')

print('Data Loaded:')
print(f'Train Multi: {Z_tr.shape}  | Val: {Z_val.shape}  | Test: {Z_test.shape}')

Data Loaded:
Train Multi: (155528, 981)  | Val: (33328, 981)  | Test: (33328, 981)


## Step 2 — Define Subsets & Train MLPs

In [2]:
from config.config_loader import load_config
from src.models.mlp_classifier import MLPTrainer
from sklearn.metrics import roc_auc_score

cfg = load_config()

sub_models = {
    'Full [0:974]':               slice(0, 974),
    'Base HC+CNN [0:718]':        slice(0, 718),
    'CNN-Only [206:718]':         slice(206, 718),
    'Handcrafted-Only [0:206]':   slice(0, 206),
    'Extended Physics [718:974]': slice(718, 974)
}

results = {}
trained_models = {}

print('\nStarting MLP Ablation Training...')
for name, slc in sub_models.items():
    print(f'\n[ {name} ]')
    z_tr_sub  = Z_tr[:, slc]
    z_val_sub = Z_val[:, slc]
    
    trainer = MLPTrainer(cfg=cfg, input_dim=z_tr_sub.shape[1])
    ckpt_name = f'mlp_abl_{name.split(" ")[0].lower()}.pt'
    
    t0 = time.time()
    best_auc = trainer.train(
        X_train=z_tr_sub, y_train=y_tr,
        X_val=z_val_sub, y_val=y_val,
        checkpoint_dir=str(CKPT_DIR),
        run_name=f'abl_{name.split(" ")[0].lower()}'
    )
    print(f'  \u2713 Trained in {time.time()-t0:.1f}s | Val AUC = {best_auc:.4f}')
    trained_models[name] = trainer


Starting MLP Ablation Training...

[ Full [0:974] ]

  MLP Training  [974-dim → 256 → 64 → 1]
  Device: cuda   Epochs: 50   Batch: 32
  Train: 155528   Val: 33328
   Epoch    T-Loss    V-Loss    T-Acc    V-Acc    V-AUC
  -------------------------------------------------------
       1    0.4022    0.2874    80.6%    87.4%   0.943  ← best
       2    0.2778    0.2751    87.8%    87.6%   0.965  ← best
       3    0.2319    0.2140    90.2%    90.9%   0.972  ← best
       4    0.2060    0.2036    91.3%    91.4%   0.975  ← best
       5    0.1835    0.1988    92.4%    91.7%   0.978  ← best
       6    0.1699    0.1912    93.0%    91.8%   0.979  ← best
       7    0.1547    0.1634    93.8%    93.2%   0.982  ← best
       8    0.1455    0.1599    94.1%    93.5%   0.984  ← best
       9    0.1379    0.1592    94.5%    93.4%   0.984  ← best
      10    0.1297    0.1546    94.9%    93.5%   0.985  ← best
      11    0.1251    0.1593    95.1%    93.7%   0.984
      12    0.1173    0.1438    95.4%

## Step 3 — Cross-Domain Evaluation & Benchmarking
Note: Comparisons are against the multi-dataset trained baseline (NB09), explicitly NOT NB03 which lacked FF++ exposure natively.

In [3]:
baseline_results = {
    'Multi': None, 
    'CelebDF': None, 
    'FF++': None, 
    'SD': None
}

for name, trainer in trained_models.items():
    slc = sub_models[name]
    
    # Evaluate AUC natively for Multi-Test, CelebDF-Test, FF++
    m_multi = trainer.evaluate(Z_test[:, slc], y_test)
    m_cd    = trainer.evaluate(Z_cd[:, slc], y_cd)
    m_ff    = trainer.evaluate(Z_ff[:, slc], y_ff)
    
    # Evaluate Stable Diffusion exclusively via detection rate to block ValueError on fake-only array
    sd_probs = trainer.predict_proba(Z_sd[:, slc])
    sd_det   = float((sd_probs >= 0.5).mean() * 100)
    
    metrics = {
        'Multi': m_multi['auc'],
        'CelebDF': m_cd['auc'],
        'FF++': m_ff['auc'],
        'SD': sd_det
    }
    results[name] = metrics
    
    # Capture 'Full' baseline to report deltas efficiently below
    if name == 'Full [0:974]':
        baseline_results = metrics

print('=' * 88)
print(f'{"Physics Benchmark MLP Ablation":^88}')
print('=' * 88)
print(f'{"Sub-Model":<30} | {"Test Multi":<12} | {"CelebDF AUC":<12} | {"FF++ AUC":<12} | {"SD Det(%)":<12}')
print('-' * 88)

for name, m in results.items():
    # Format baseline deltas exclusively for visual distinction
    if name == 'Full [0:974]':
        print(f'{name:<30} | {m["Multi"]:<12.4f} | {m["CelebDF"]:<12.4f} | {m["FF++"]:<12.4f} | {m["SD"]:<11.1f}%')
        print('-' * 88)
    else:
        d_m = m['Multi'] - baseline_results['Multi']
        d_cd = m['CelebDF'] - baseline_results['CelebDF']
        d_ff = m['FF++'] - baseline_results['FF++']
        d_sd = m['SD'] - baseline_results['SD']
        print(f'{name:<30} | {m["Multi"]:.4f} ({d_m:+.4f}) | {m["CelebDF"]:.4f} ({d_cd:+.4f}) | {m["FF++"]:.4f} ({d_ff:+.4f}) | {m["SD"]:.1f}% ({d_sd:+.1f}%)')
print('=' * 88)

                             Physics Benchmark MLP Ablation                             
Sub-Model                      | Test Multi   | CelebDF AUC  | FF++ AUC     | SD Det(%)   
----------------------------------------------------------------------------------------
Full [0:974]                   | 0.9924       | 0.9995       | 0.9969       | 100.0      %
----------------------------------------------------------------------------------------
Base HC+CNN [0:718]            | 0.9894 (-0.0030) | 0.9993 (-0.0002) | 0.9957 (-0.0012) | 100.0% (+0.0%)
CNN-Only [206:718]             | 0.9921 (-0.0002) | 0.9994 (-0.0000) | 0.9963 (-0.0006) | 100.0% (+0.0%)
Handcrafted-Only [0:206]       | 0.7835 (-0.2088) | 0.8552 (-0.1442) | 0.7651 (-0.2318) | 99.2% (-0.8%)
Extended Physics [718:974]     | 0.9818 (-0.0106) | 0.9975 (-0.0020) | 0.9864 (-0.0105) | 100.0% (-0.0%)
